# 03 — Forecasting Model Selection
Validates ForecastAgent's three models on the IoT time series dataset
and measures how accurately `model_selector.select_best_model()` chooses
the right model for different data characteristics.

**Real-time integration**: ForecastAgent runs Prophet + AutoARIMA concurrently
via `asyncio.gather()`. This notebook measures wall-clock time for each model
and validates that parallel execution stays under the 5-minute pipeline budget.


In [1]:
import sys, time, asyncio
sys.path.insert(0, '..')
import polars as pl
import pandas as pd
import numpy as np

iot = pl.read_parquet('datasets/sample_iot_timeseries.parquet')
print(f"IoT dataset: {iot.shape}")
print(f"Date range: {iot['timestamp'].min()} to {iot['timestamp'].max()}")
print(f"Columns: {iot.columns}")


IoT dataset: (8760, 10)
Date range: 2024-01-01 00:00:00 to 2024-12-30 23:00:00
Columns: ['timestamp', 'sensor_id', 'temperature_c', 'power_kw', 'vibration_mm_s', 'humidity_pct', 'pressure_hpa', 'hour_of_day', 'day_of_week', 'is_business_hours']


## Data preparation — create clean training series

In [2]:
# Prepare a clean temperature series for forecasting
df_temp = (
    iot
    .select(['timestamp', 'temperature_c'])
    .drop_nulls()
    .sort('timestamp')
    # Remove the 3 injected anomalies before training
    .filter(pl.col('temperature_c').is_between(-10, 60))
)
print(f"Training rows after cleaning: {len(df_temp)}")
print(f"Temperature stats: mean={df_temp['temperature_c'].mean():.1f}°C, "
      f"std={df_temp['temperature_c'].std():.1f}°C")

# Use last 30 days as holdout for MAPE evaluation
cutoff     = df_temp['timestamp'].max() - pd.Timedelta(days=30)
train_df   = df_temp.filter(pl.col('timestamp') < cutoff)
holdout_df = df_temp.filter(pl.col('timestamp') >= cutoff)
print(f"Train: {len(train_df)} rows | Holdout: {len(holdout_df)} rows")


Training rows after cleaning: 8582
Temperature stats: mean=20.0°C, std=6.1°C
Train: 7874 rows | Holdout: 708 rows


## Model 1: Prophet

In [3]:
from importlib import import_module

run_prophet = import_module("backend.agents.analysis.forecast.prophet_runner").run_prophet

print("Running Prophet (horizon=30 days)...")
t0 = time.perf_counter()
prophet_result = await run_prophet(train_df, 'timestamp', 'temperature_c', horizon=30)
prophet_ms = int((time.perf_counter() - t0) * 1000)

if 'error' in prophet_result:
    print(f"Prophet failed: {prophet_result['error']}")
else:
    preds = prophet_result['predictions']
    print(f"  Model:      {prophet_result['model']}")
    print(f"  Duration:   {prophet_ms}ms")
    print(f"  Trend:      {prophet_result['trend_direction']}")
    print(f"  Predictions: {len(preds)} days")
    print(f"  Sample forecast: {preds[0]['timestamp']} → {preds[0]['value']:.2f}°C "
          f"[{preds[0]['lower_bound']:.2f}, {preds[0]['upper_bound']:.2f}]")


Running Prophet (horizon=30 days)...


c:\Users\manit\OneDrive\Desktop\projects\Data-Analyst-Agent\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


2026-07-06 22:18:02 [info     ] prophet_complete               horizon=30 rows_trained=7874 trend=down
  Model:      Prophet
  Duration:   4913ms
  Trend:      down
  Predictions: 30 days
  Sample forecast: 2024-12-01 → 0.12°C [-4.17, 4.38]


## Model 2: AutoARIMA

In [4]:
from importlib import import_module

run_arima = import_module("backend.agents.analysis.forecast.arima_runner").run_arima

print("Running AutoARIMA (horizon=30 days)...")
t0 = time.perf_counter()
arima_result = await run_arima(train_df, 'timestamp', 'temperature_c', horizon=30)
arima_ms = int((time.perf_counter() - t0) * 1000)

if 'error' in arima_result:
    print(f"AutoARIMA failed: {arima_result['error']}")
else:
    preds = arima_result['predictions']
    print(f"  Model:      {arima_result['model']}")
    print(f"  Duration:   {arima_ms}ms")
    print(f"  Predictions: {len(preds)} days")
    print(f"  Sample: {preds[0]['timestamp']} → {preds[0]['value']:.2f}°C")


Running AutoARIMA (horizon=30 days)...
2026-07-06 22:20:10 [info     ] arima_complete                 horizon=30 rows_trained=7874 season_length=7
  Model:      AutoARIMA
  Duration:   127207ms
  Predictions: 30 days
  Sample: 2024-12-01 → 14.68°C


## Model 3: XGBoost

In [5]:
from importlib import import_module

run_xgboost = import_module("backend.agents.analysis.forecast.xgboost_forecaster").run_xgboost

print("Running XGBoost (horizon=30 days)...")
t0 = time.perf_counter()
xgb_result = await run_xgboost(train_df, 'timestamp', 'temperature_c', horizon=30)
xgb_ms = int((time.perf_counter() - t0) * 1000)

if 'error' in xgb_result:
    print(f"XGBoost failed: {xgb_result['error']}")
else:
    preds = xgb_result['predictions']
    print(f"  Model:          {xgb_result['model']}")
    print(f"  Duration:       {xgb_ms}ms")
    print(f"  Predictions:    {len(preds)} days")
    top_feat = max(xgb_result['feature_importances'], key=xgb_result['feature_importances'].get)
    print(f"  Top feature:    {top_feat} ({xgb_result['feature_importances'][top_feat]:.3f})")


Running XGBoost (horizon=30 days)...
2026-07-06 22:20:11 [info     ] xgboost_complete               horizon=30 rows_trained=7853 top_feature=lag_1
  Model:          XGBoost
  Duration:       1701ms
  Predictions:    30 days
  Top feature:    lag_1 (0.926)


## MAPE Evaluation on holdout

In [6]:
from backend.agents.analysis.forecast.model_selector import compute_mape

if holdout_df is not None and len(holdout_df) > 0:
    actual = holdout_df['temperature_c'].to_list()[:30]

    for name, result in [('Prophet', prophet_result), ('AutoARIMA', arima_result), ('XGBoost', xgb_result)]:
        if 'error' in result:
            print(f"{name}: FAILED")
            continue
        predicted = [p['value'] for p in result['predictions'][:len(actual)]]
        mape = compute_mape(actual, predicted)
        print(f"{name}: MAPE = {mape:.4f} ({mape:.1%})" if mape else f"{name}: MAPE undefined")


Prophet: MAPE = 1.3191 (131.9%)
AutoARIMA: MAPE = 0.0948 (9.5%)
XGBoost: MAPE = 0.0400 (4.0%)


## Parallel execution — asyncio.gather() vs sequential

In [7]:
print("Sequential execution time:")
t0 = time.perf_counter()
_ = await run_prophet(train_df, 'timestamp', 'temperature_c', 30)
_ = await run_arima(train_df, 'timestamp', 'temperature_c', 30)
seq_ms = int((time.perf_counter() - t0) * 1000)
print(f"  Sequential: {seq_ms}ms")

print("\nParallel execution time (asyncio.gather):")
async def run_parallel():
    return await asyncio.gather(
        run_prophet(train_df, 'timestamp', 'temperature_c', 30),
        run_arima(train_df, 'timestamp', 'temperature_c', 30),
    )

t0 = time.perf_counter()
results = await run_parallel()
par_ms = int((time.perf_counter() - t0) * 1000)
print(f"  Parallel:   {par_ms}ms")
print(f"  Speedup:    {seq_ms/par_ms:.1f}x (wall-clock time saved for real-time pipeline)")


Sequential execution time:
2026-07-06 22:20:16 [info     ] prophet_complete               horizon=30 rows_trained=7874 trend=down
2026-07-06 22:21:54 [info     ] arima_complete                 horizon=30 rows_trained=7874 season_length=7
  Sequential: 102660ms

Parallel execution time (asyncio.gather):
2026-07-06 22:22:07 [info     ] prophet_complete               horizon=30 rows_trained=7874 trend=down
2026-07-06 22:23:29 [info     ] arima_complete                 horizon=30 rows_trained=7874 season_length=7
  Parallel:   95003ms
  Speedup:    1.1x (wall-clock time saved for real-time pipeline)


## Model selector validation

In [8]:
from backend.agents.analysis.forecast.model_selector import select_best_model

best = select_best_model(prophet_result, arima_result, xgb_result)
print(f"Selected model: {best.get('model')}")
print(f"Reason: {'MAPE-based' if best.get('mape') else 'Priority-based (Prophet > ARIMA > XGBoost)'}")
print(f"Trend direction: {best.get('trend_direction', 'unknown')}")


2026-07-06 22:23:29 [info     ] model_selected_by_priority     model=Prophet
Selected model: Prophet
Reason: Priority-based (Prophet > ARIMA > XGBoost)
Trend direction: down
